⚠️ This is my personal problem formulation, I am still working on it,


If you are reading this and you are not **LARIBI Issam Abdellah** and you want to commit you own problem formulation use a different notebook on a folder with you name.



## Problem Formulation

### 1. State Representation

```python
state = {
    'robots': [
        {
            'id': 1,
            'position': (x, y),
            'goal': (gx, gy),
            'path': [(x1, y1), (x2, y2), ...],
            'path_index': 0,
            'at_goal': False
        },
        # ... more robots
    ],
    'time_step': 0,
    'collisions': 0,
    'deadlock': False,
    'grid': GridEnvironment
}
```

**State Variables:**
- `time_step`: Current simulation time step (0, 1, 2, ...)
- `collisions`: Total number of conflicts detected so far
- `deadlock`: True if robots are stuck (no progress made)
- `grid`: Reference to GridEnvironment object

**Constraints in State:**
- ...
- ...

### 2. Actions

- **PlanPaths(Algorithm, Robots[])**: Use A* or any other search algorithm to find paths
- **ExecuteStep(Robots[])**: All robots move simultaneously (one step each)
- **MoveRobot(Robot, Direction)**: Robot moves one step along planned path
- **Wait(Robot, Steps)**: Robot does not move for specified number of steps

Note: the directions in the grid are: up/down/left/right Hence there are actions goUp goDown...etc

⚠️ Here we are in a multi-agent system, i'm not sure of these ations!


### 3. Goal Test


**Primary Goal**: All robots at their goal positions

**Goal Conditions:**
- `robot['position'] == robot['goal']` for all robots
- `robot['at_goal'] == True` for all robots
- `time_step` is minimized (makespan criterion)

**Success Criteria:**
1. All robots reach goals 
2. No collisions/conflicts 
3. No deadlocks 
4. Minimum makespan  (secondary objective)
5. Minimum flowtime  (secondary objective)


### 4. Path Cost

- **Makespan**: Time until last robot finishes = max(len(path) for all robots)
- **Flowtime**: Sum of all path lengths = sum(len(path) for all robots)
- Each move costs: 1 time unit
- Eech wait costs: 1 time unit
- Collision penalty: ??? (1)
- Deadlock penalty: ??? (2)

⚠️ For (1) and (2) maybe we run several tests to determine the best values

### Initial State

```python
initial_state = {
    'robots': [
        {'id': 1, 'position': (1, 1), 'goal': (8, 8), 'path': [], 'path_index': 0, 'at_goal': False},
        {'id': 2, 'position': (8, 1), 'goal': (1, 8), 'path': [], 'path_index': 0, 'at_goal': False},
        {'id': 3, 'position': (1, 8), 'goal': (8, 1), 'path': [], 'path_index': 0, 'at_goal': False},
    ],
    'time_step': 0,
    'collisions': 0,
    'deadlock': False,
    'grid': GridEnvironment(filename)
}
```

**Initial State Example:**
- Robot 1: starts at (1,1), goal (8,8)
- Robot 2: starts at (8,1), goal (1,8)
- Robot 3: starts at (1,8), goal (8,1)
- All paths are empty
- Grid loaded from the file

**Properties of Initial State:**
- All robots at distinct start positions
- All start positions are walkable
- No paths planned yet
- No conflicts exist initially
- Time counter starts at 0
- Cost g(n) = 0 (no moves yet)

## Problem Definition

Given N robots at start positions and N goal positions (shelves to retrieve), find a set of
conflict-free paths such that all robots reach their goals in minimal time.

## Problem Formulation
### 1. State Representation

Each state is represented by a dictionary with the following structure:

```python
state = {
    'time_step': t,                           # Current simulation time (0, 1, 2, ...)
    
    # Robot configurations at time t
    'robots': [
        {
            'id': robot_id,                             # Unique identifier (0, 1, 2, ...)
            'start_position': (x0, y0),                 # Start postion (fixed)
            'goal_position': (gx, gy),                  # Goal position (fixed)
            'path': [(x0,y0), (x1,y1), ..., (gx, gy)],  # Full planned path (computed)
            'path_index': i,                            # Current position in path (1 initialy)
            'current_position': path[i],                # Current position in the grid
            'at_goal': bool,                            # True if path[-1] == goal
            'arrival_time': t_arr                       # Time step when reached goal 
        },
        # ... more robots
    ],
    
    # Global state
    'positions': {robot_id: (x, y)},          # Quick lookup: robot_id → position
    'collisions': count,                      # Cumulative collision count
    'deadlock_detected': bool,                # True if circular wait detected
    'grid': GridEnvironment                   # Reference to warehouse layout
}
```

### 2. Goal Test

**Primary Goal**:
- No collisions occurred
- No deadlock detected
- All robots at their goal positions


**Success Criteria: (Detail)**
1. All robots reach goals 
2. No collisions/conflicts 
3. No deadlocks 
4. Minimum makespan  (secondary objective)
5. Minimum flowtime  (secondary objective)

### 3. Actions

In this multi-agent system, we have three types of actions:


1. **Planning actions**: computes paths for all the robots in the grid  
    - IndependentPlanning: uses Independent A* search algorithm to plan paths for all robots
    - CooperativePLanning: uses Cooperative A* search algorithm to plan paths for all robots

2. **Optimizing actions**:
    - HillClimbingOptimization: Take a valid solution and try to shorten paths by swapping
move orders

3. **Execution actions**: After paths are planned, we start the execution
    - ExecuteStep: all robots move simultaneously according to their planned paths


    - SubActions per robot: 
        - MoveUp: move up if walkable (path = [..., (x, y), (x - 1, y), ...]) 


        - MoveDown: move down if walkable (path = [..., (x, y), (x + 1, y), ...]) 


        - MoveLeft: move left if walkable (path = [..., (x, y), (x, y - 1), ...]) 


        - MoveRight: move right if walkable (path = [..., (x, y), (x, y + 1), ...]) 


        - Wait: stay in current position (path = [..., (x, y), (x, y), ...]) 



### 4. Transition Model

- Planning Actions Transition: All the robots now have their paths planned

- Optimizing Actions Transition: Some/All robots got their paths updated

- Execution Actions Transitions: 
    - The time step of the state changes
    - The path index for all robots changes
    - The positions of the robots changes



### 5. Path Cost

- Each move costs: 1 time unit
- Each wait costs: 1 time unit


### Initial State


**Properties of Initial State:**
- All robots at distinct start positions
- All start positions are walkable
- No paths planned yet
- No conflicts exist initially
- Time counter starts at 0
- Cost g(n) = 0 (no moves yet)


**Initial State Example:**
```python
initial_state = {
    'time_step': 0,
    'robots': [
        {'id': 1, 'start_position': (1, 1), 'goal_position': (8, 8), 'path': [(1, 1)], 'path_index': 0, 'at_goal': False, 'current position': path[path_index], 'arrival_time': None},
        {'id': 2, 'start_position': (8, 1), 'goal_position': (1, 8), 'path': [(8, 1)], 'path_index': 0, 'at_goal': False, 'current position': path[path_index], 'arrival_time': None},
        {'id': 3, 'start_position': (1, 8), 'goal_position': (8, 1), 'path': [(1, 8)], 'path_index': 0, 'at_goal': False, 'current position': path[path_index], 'arrival_time': None},
    ],
    'positions': {1: (1, 1), 2: (8, 1), 3: (1, 8)}
    'collisions': 0,
    'deadlock': False,
    'grid': GridEnvironment(filename)
}
```

- time_step is 0
- Robot 1: starts at (1,1), goal (8,8)
- Robot 2: starts at (8,1), goal (1,8)
- Robot 3: starts at (1,8), goal (8,1)
- Robots' positions are their starting positions
- No collisions 
- No deadlock
- Grid loaded from the file
